# Infinite Series Summation using Euler–Maclaurin and Euler–Boole Tail Approximations

This notebook demonstrates the computation of various infinite series sums using tail approximations. Each example is run in its own cell so you can capture the output (both numerical and plots) for your Thesis.

In [ ]:
# Install and update required packages
import Pkg
Pkg.activate(".")
Pkg.instantiate()

# Add/update required packages
Pkg.add("SymPy")
Pkg.add("Plots")
Pkg.add("Printf")
Pkg.add("SpecialFunctions")
Pkg.update()

## Module Definition: Analytictail

The following cell defines the module `Analytictail` which contains all functions needed for computing the infinite series sums using Euler–Maclaurin (for non‐alternating series) and Euler–Boole (for alternating series) tail approximations. Note that we export the symbols `run_example` and `x_` so they can be used directly from the Main module.

In [ ]:
module Analytictail

using SymPy
using Plots
using Printf
using SpecialFunctions  # For coth, etc.

export run_example, sum_infinite_series, plot_error_vs_m, x_

##########################################################
# 1) Setup
##########################################################
setprecision(256)  # Very high precision for demonstration
gr()

# Partial-sum cutoffs
const Kset = [1, 2, 3, 4]
# Max correction terms
const m_max = 30

# We'll store Bernoulli / Euler polynomials in caches
const bernoulliCache = Dict{Int, Vector{Rational{BigInt}}}()
const eulerPolyCache = Dict{Tuple{Int, Float64}, Dict{Int, Float64}}()

##########################################################
# 2) Bernoulli numbers (caching)
##########################################################
function BernoulliListCached(n::Int)::Vector{Rational{BigInt}}
    if haskey(bernoulliCache, n)
        return bernoulliCache[n]
    end
    A = Vector{Rational{BigInt}}(undef, n+1)
    B = similar(A)
    for mm in 0:n
        A[mm+1] = 1 // (mm+1)
        for j in mm:-1:1
            A[j] = j*(A[j]-A[j+1])
        end
        B[mm+1] = A[1]
    end
    bernoulliCache[n] = B
    return B
end

##########################################################
# 3) Euler polynomials (for Euler–Boole)
##########################################################
function euler_polynomials_cached(max_n::Int, x_val::Float64)::Dict{Int, Float64}
    key = (max_n, x_val)
    if haskey(eulerPolyCache, key)
        return eulerPolyCache[key]
    end

    t = SymPy.symbols("t")
    expr = (2*SymPy.exp(x_val*t)) / (SymPy.exp(t)+1)
    out = Dict{Int, Float64}()
    for n in 0:max_n
        try
            ser = expr.series(t, 0, n+1).expand()
            coeff_n = factorial(big(n))*ser.coeff(t,n)
            out[n] = float(coeff_n)
        catch
            @warn "Failed Euler poly E_$n($x_val)"
            out[n] = 0.0
        end
    end
    eulerPolyCache[key] = out
    return out
end

##########################################################
# 4) Utility: safe_real_float
##########################################################
function safe_real_float(expr)::Float64
    try
        val = expr.evalf()
        return float(SymPy.real(val))
    catch
        return NaN
    end
end

##########################################################
# 5) Euler–Boole tail approximation (ALTERNATING)
##########################################################
function euler_boole_infinite_tail(f::Sym, m::Int)::Float64
    local x_ = SymPy.symbols("x")
    f1 = safe_real_float(f.subs(x_, 1))

    e_dict = euler_polynomials_cached(m, 1.0)
    sum_corr = 0.0
    for j in 1:m
        d_expr = diff(f, x_, j).subs(x_, 1)
        d_val  = safe_real_float(d_expr)
        e_j    = e_dict[j]
        sum_corr += (e_j / factorial(big(j))) * d_val
    end
    return 0.5*f1 + sum_corr
end

##########################################################
# 6) Euler–Maclaurin tail approximation (NON-ALTERNATING)
##########################################################
function euler_maclaurin_infinite_tail(f::Sym, m::Int)::Float64
    local x_ = SymPy.symbols("x")
    
    integral_sym = try
        SymPy.integrate(f, (x_, 1, SymPy.oo))
    catch
        nothing
    end

    if integral_sym === nothing || integral_sym == SymPy.oo || integral_sym == -SymPy.oo
        @warn "Analytical integration failed or diverged; setting integral to 0.0."
        integral_val = 0.0
    else
        integral_val = safe_real_float(integral_sym)
    end

    half_term = 0.5 * safe_real_float(f.subs(x_, 1))
    
    Blist = BernoulliListCached(2*m)
    sum_corr = 0.0
    for j in 1:m
        dorder = 2*j - 1
        dexpr  = diff(f, x_, dorder).subs(x_, 1)
        dval   = safe_real_float(dexpr)
        b_2j   = float(Blist[2*j+1])
        denom  = factorial(big(2*j))
        sum_corr += (b_2j / denom) * dval
    end

    return integral_val + half_term - sum_corr
end

##########################################################
# 7) sum_infinite_series = partial sum + tail
##########################################################
function sum_infinite_series(f::Sym, k::Int, m::Int, alt::Bool)::Float64
    local x_ = SymPy.symbols("x")

    S_k = 0.0
    for n in 1:k
        S_k += safe_real_float(f.subs(x_, n))
    end

    f_tail = f.subs(x_, x_ + k)

    if alt
        return S_k + euler_boole_infinite_tail(f_tail, m)
    else
        return S_k + euler_maclaurin_infinite_tail(f_tail, m)
    end
end

##########################################################
# 8) plot_error_vs_m
##########################################################
function plot_error_vs_m(f::Sym, desc::String, known_val::Float64, alt::Bool)
    p = plot(
        title = desc * " : log10(|error|) vs. m",
        xlabel = "m (derivative expansions)",
        ylabel = "log10(|error|)",
        legend = :topright
    )

    for k in Kset
        local logerrs = Float64[]
        println("\n$desc => alt=$alt, k=$k => exact=$known_val")
        println("m\tApprox\tError")
        for mm in 1:m_max
            approx = sum_infinite_series(f, k, mm, alt)
            err = abs(approx - known_val)
            @printf("%2d\t%.16e\t%.16e\n", mm, approx, err)
            push!(logerrs, log10(max(err,1e-300)))
        end
        plot!(p, 1:m_max, logerrs, label="k=$k", linewidth=2)
    end

    display(p)
end

##########################################################
# 9) run_example
##########################################################
function run_example(f::Sym, desc::String, known_val::Float64, alt::Bool)
    println("\n=== run_example: $desc")
    plot_error_vs_m(f, desc, known_val, alt)
end

# Export x_ for use in Main
const x_ = SymPy.symbols("x")

end # module

In [ ]:
# Import the module into Main
using .Analytictail

In [ ]:
# Import SymPy into Main so that `Sym` is defined
using SymPy

In [ ]:
# Setup common constants for examples
zeta5_approx = 1.03692775514337
eta3_approx = 0.9015426773696955
alt_sqrt_approx = -0.6048986434

### Example 1: Euler Macheroni: sum(1/n)

In [ ]:
run_example(1/x_, "Euler Macheroni: sum(1/n)", 0.57721, false)

### Example 2: Basel: sum(1/n²) = π²/6

In [ ]:
run_example(1/x_^2, "Basel: sum(1/n²) = π²/6", float(pi^2/6), false)

### Example 3: Zeta(3): sum(1/n³) ≈ 1.20206

In [ ]:
run_example(1/x_^3, "Zeta(3): sum(1/n³) ≈ 1.20206", 1.202056903159594, false)

### Example 4: Zeta(4): sum(1/n⁴) = π⁴/90

In [ ]:
run_example(1/x_^4, "Zeta(4): sum(1/n⁴) = π⁴/90", float(pi^4/90), false)

### Example 5: Zeta(5): sum(1/n⁵) ≈ 1.03693

In [ ]:
run_example(1/x_^5, "Zeta(5): sum(1/n⁵) ≈ 1.03693", zeta5_approx, false)

### Example 6: Zeta(6): sum(1/n⁶) = π⁶/945

In [ ]:
run_example(1/x_^6, "Zeta(6): sum(1/n⁶) = π⁶/945", float(pi^6/945), false)

### Example 7: sum(e⁻ⁿ): 1/(e-1)

In [ ]:
run_example(exp(-x_), "sum(e⁻ⁿ): 1/(e-1)", 1/(exp(1)-1), false)

### Example 8: sum(2⁻ⁿ) = 1

In [ ]:
run_example((Sym(2))^(-x_), "sum(2⁻ⁿ) = 1", 1.0, false)

### Example 9: sum(1/(n+1)²) = ζ(2)-1

In [ ]:
run_example(1/(x_+1)^2, "sum(1/(n+1)²) = ζ(2)-1", float(pi^2/6 - 1.0), false)

### Example 10: sum(1/[n(n+1)]) = 1

In [ ]:
run_example(1/(x_*(x_+1)), "sum(1/[n(n+1)]) = 1", 1.0, false)

### Example 11: sum[(-1)^(n+1)/n] = ln2

In [ ]:
run_example(-cos(pi*x_)/x_, "sum[(-1)^(n+1)/n] = ln2", log(2), true)

### Example 12: sum[(-1)^(n+1)/n²] = π²/12

In [ ]:
run_example(-cos(pi*x_)/x_^2, "sum[(-1)^(n+1)/n²] = π²/12", float(pi^2/12), true)

### Example 13: sum[(-1)^(n+1)/n³] ≈ 0.90154

In [ ]:
run_example(-cos(pi*x_)/x_^3, "sum[(-1)^(n+1)/n³] ≈ 0.90154", eta3_approx, true)

### Example 14: Catalan: ≈ 0.91597

In [ ]:
run_example(cos(pi*(x_-1))/(2*x_ - 1)^2, "Catalan: ≈ 0.91597", 0.915965594177219, true)

### Example 15: Arctan(1): π/4

In [ ]:
run_example(cos(pi*(x_-1))/(2*x_ - 1), "Arctan(1): π/4", float(pi/4), true)

### Example 16: sum[(-1)^n/√n] ≈ -0.60490

In [ ]:
run_example(cos(pi*x_)/sqrt(x_), "sum[(-1)^n/√n] ≈ -0.60490", alt_sqrt_approx, true)